<img src="../img/GTK_Logo_Social_Icon.jpg" width=175 align="right" />

# Worksheet 12.1: Building & Using an MCP Server

In Module 11 you hand-wrote tools and bound them directly to one agent. The
**Model Context Protocol (MCP)** lets tools live in a separate *server* that any
MCP-aware application can connect to.

In this lab you connect an agent to a small security-tools MCP server, load its
tools over MCP, and let the agent investigate emails. The server
(`mcp_server.py`) is given to you; your job is the **client side**: connecting,
loading the tools, and wiring them into an agent.

Docs: [langchain-mcp-adapters](https://github.com/langchain-ai/langchain-mcp-adapters) · [Model Context Protocol](https://modelcontextprotocol.io/)


In [ ]:
# Load Libraries - Make sure to run this cell!
import os
import sys
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

# Loads ANTHROPIC_API_KEY from your .env file in the project root.
load_dotenv()

MODEL = "claude-opus-4-8"

# Find mcp_server.py whether this notebook runs from notebooks/, answers/,
# or the repo root, so the path never depends on your working directory.
SERVER_PATH = next(
    (os.path.abspath(p) for p in
     ("mcp_server.py", "../notebooks/mcp_server.py", "notebooks/mcp_server.py")
     if os.path.exists(p)), None)
assert SERVER_PATH, "Could not find mcp_server.py — run this from the notebooks/ folder."

## The server you'll connect to

Open **`mcp_server.py`** (in this folder) and skim it. It is a complete MCP
server built with `FastMCP`, and it exposes three simulated security tools:

- `check_url_reputation` — reputation of a URL or domain
- `check_sender_reputation` — reputation of a sender address
- `check_email_authentication` — SPF / DKIM / DMARC analysis

You do not need to edit the server. An MCP server is just a program that exposes
tools over a standard protocol; the host launches it and your agent calls its
tools without knowing how they are implemented.


## Step 1 — Connect a client to the server

`MultiServerMCPClient` takes a dictionary describing each server: the command to
launch it, its arguments, and the transport. Because the server runs locally, we
launch it as a subprocess and talk to it over **stdio**.


In [ ]:
# TODO: complete the connection config below.
# Hint: the server runs locally as a subprocess, so the transport is "stdio".
server_config = {
    "security-tools": {
        "command": sys.executable,   # the same Python running this notebook
        "args": [SERVER_PATH],       # the server file (resolved in the setup cell)
        "transport": "____",
    }
}

client = MultiServerMCPClient(server_config)
print("Client configured for the security-tools server.")

## Step 2 — Load the server's tools

`get_tools()` asks the server what it offers and hands the tools back as
LangChain tools that your agent can call. It is asynchronous, so we `await` it.


In [ ]:
# TODO: load the tools from the server (this call is async).
# Hint: the client method is get_tools().
tools = await client.____()

for t in tools:
    print(f"{t.name}: {t.description.splitlines()[0]}")

assert len(tools) == 3, "Expected 3 tools from the security-tools server"

## Step 3 — Build an agent with those tools

Hand the tools to `create_agent`, exactly as in Module 12. The agent does not
care that these tools live in a separate MCP server rather than in this notebook.


In [ ]:
SYSTEM_PROMPT = (
    "You are a SOC phishing-triage assistant. Investigate the email using the "
    "available tools, then give a final verdict of 'phishing', 'suspicious', or "
    "'legitimate' with one or two sentences of reasoning."
)

llm = ChatAnthropic(model=MODEL)

# TODO: create the agent from the llm and the tools you loaded in Step 2.
agent = create_agent(____, ____, system_prompt=SYSTEM_PROMPT)

## Step 4 — Investigate a phishing email

Send the agent an email and watch it decide which tools to call. It should reach
a `phishing` verdict on its own.


In [ ]:
PHISHING_EMAIL = """From: PayPal Billing <billing@paypa1-secure.com>
Authentication results: SPF=fail, DKIM=none, DMARC=fail

Your account has been locked for suspicious activity. Verify your identity
immediately at http://paypa1-secure.com/login or your account will be closed.
"""

# TODO: send the email to the agent. Fill in the user message content.
result = await agent.ainvoke({"messages": [("user", ____)]})
print(result["messages"][-1].content)

## Step 5 — Now a legitimate email

Run the same investigation on a real message. The agent should call the tools,
see clean authentication and a trusted domain, and return `legitimate`.


In [ ]:
LEGIT_EMAIL = """From: GitHub <noreply@github.com>
Authentication results: SPF=pass, DKIM=pass, DMARC=pass

A new sign-in to your account from Chrome on macOS. If this was you, no action
is needed. You can review activity at https://github.com/settings/security
"""

# TODO: investigate the legitimate email the same way.
result = await agent.ainvoke({"messages": [("user", ____)]})
print(result["messages"][-1].content)

## What you built

You connected an agent to tools it does not contain. The same agent pattern from
Module 13 now uses tools that live in a separate, swappable MCP server. Point the
client at a different server and the agent gains new abilities with no change to
its own code.

**Security reminder:** every MCP server you connect is code you are trusting with
your agent's context and actions. Vet and pin the servers you use, give each one
the least access it needs, and keep a human in the loop for anything destructive.
